In [111]:
import os
import joblib
import numpy as np
import nest_asyncio
import uvicorn
from IPython.display import Markdown, display
from fastapi import FastAPI
from pydantic import BaseModel
from dotenv import load_dotenv
import re
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
import os
nest_asyncio.apply()
from fastapi.responses import FileResponse
load_dotenv()

GEMINI_API_KEY = os.getenv("geminiapi")
GROQ_API_KEY = os.getenv("groqapi")
DEEPSEEK_API_KEY = os.getenv("deepseekapi")

print("Environment variables loaded successfully!")

Environment variables loaded successfully!


In [112]:
MODEL_PATH = "models/anemia_random_forest_model.pkl"
FEATURE_PATH = "models/model_features.pkl"

model = joblib.load(MODEL_PATH)
features = joblib.load(FEATURE_PATH)

print("Model and features loaded successfully!")
print("Feature order:", features)

Model and features loaded successfully!
Feature order: ['Gender', 'Age', 'Hb', 'RBC', 'PCV', 'MCV', 'MCH', 'MCHC']


In [113]:
def build_medical_prompt(patient, prediction, confidence):
    return f"""
You are a professional clinical AI assistant specialized in hematology.

Patient CBC Data:
- Name: {patient['name']}
- Phone: {patient['phone']}
- Age: {patient['Age']}
- Gender: {patient['Gender']}
- Hemoglobin (Hb): {patient['Hb']}
- RBC: {patient['RBC']}
- PCV: {patient['PCV']}
- MCV: {patient['MCV']}
- MCH: {patient['MCH']}
- MCHC: {patient['MCHC']}

ML Model Prediction: {"Anemic" if prediction == 1 else "Not Anemic"}
Confidence Score: {confidence:.2f}%

IMPORTANT FORMAT INSTRUCTIONS:

1. FIRST generate a structured clinical table comparing patient values with normal reference ranges and beofore that add person age , name and phone in the report.
2. THEN provide:
   - Clinical Interpretation
   - Risk Level (Low/Moderate/High)
   - Possible Anemia Type (if applicable)
   - Suggested Diagnostic Tests
3. Keep it medically accurate, professional, and a little bit bigger in length. Avoid being too concise.
don't give disclaimer at the end of the report.
The output must be well-structured using clear headings and a table.
"""

In [114]:
import google.generativeai as genai
from openai import OpenAI

genai.configure(api_key=GEMINI_API_KEY)

def gemini_report(prompt):
    try:
        model_gemini = genai.GenerativeModel("gemini-3-flash-preview")
        response = model_gemini.generate_content(
            prompt,
            generation_config={
                "temperature": 0.2,
                
            }
            
            )
        return response.text
    except Exception as e:
        return f"Gemini Error: {str(e)}"
    



def clean_html_tags(text):
    """Remove all HTML tags like <b>, <i>, etc."""
    return re.sub(r'<.*?>', '', text)

def is_separator_row(line):
    """Detect markdown separator row like | :--- | :--- |"""
    return bool(re.match(r'^\|\s*:?-+:?\s*(\|\s*:?-+:?\s*)+\|$', line))

def generate_pdf_report(report_text, filename="clinical_report.pdf"):
    doc = SimpleDocTemplate(
        filename,
        pagesize=A4,
        rightMargin=40,
        leftMargin=40,
        topMargin=40,
        bottomMargin=40
    )

    styles = getSampleStyleSheet()
    content = []

    # Title
    title = Paragraph("HemoScan AI - Clinical Hematology Report", styles["Title"])
    content.append(title)
    content.append(Spacer(1, 12))

    lines = report_text.split("\n")
    table_data = []
    inside_table = False

    for raw_line in lines:
        line = raw_line.strip()

        if not line:
            continue

        # Convert **bold** markdown to plain text bold (for paragraph rendering)
        line = re.sub(r"\*\*(.*?)\*\*", r"\1", line)

        # Clean ALL HTML tags (CRITICAL FIX)
        line = clean_html_tags(line)

        # Detect table rows
        if line.startswith("|") and line.endswith("|"):

            # 🚨 Skip markdown separator row
            if is_separator_row(line):
                continue

            inside_table = True

            # Split row cells
            cells = [clean_html_tags(cell.strip()) for cell in line.strip("|").split("|")]

            table_data.append(cells)
            continue

        else:
            # If table ended, render it
            if inside_table and table_data:
                table = Table(table_data, repeatRows=1)

                table.setStyle(TableStyle([
                    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
                    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                    ("GRID", (0, 0), (-1, -1), 0.75, colors.black),
                    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
                    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                    ("PADDING", (0, 0), (-1, -1), 6),
                ]))

                content.append(table)
                content.append(Spacer(1, 16))

                table_data = []
                inside_table = False

        # Headings
        if line.startswith("###"):
            heading = Paragraph(line.replace("###", "").strip(), styles["Heading2"])
            content.append(heading)

        elif line.startswith("####"):
            subheading = Paragraph(line.replace("####", "").strip(), styles["Heading3"])
            content.append(subheading)

        elif line == "---":
            content.append(Spacer(1, 12))

        else:
            paragraph = Paragraph(line, styles["BodyText"])
            content.append(paragraph)

    # Render table if report ends with table
    if table_data:
        table = Table(table_data, repeatRows=1)
        table.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("GRID", (0, 0), (-1, -1), 0.75, colors.black),
            ("ALIGN", (0, 0), (-1, -1), "CENTER"),
            ("PADDING", (0, 0), (-1, -1), 6),
        ]))
        content.append(table)

    doc.build(content)
    return filename



Task exception was never retrieved
future: <Task finished name='Task-108' coro=<Server.serve() done, defined at c:\Users\avina\AppData\Local\Programs\Python\Python312\Lib\site-packages\uvicorn\server.py:68> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "c:\Users\avina\AppData\Local\Programs\Python\Python312\Lib\site-packages\uvicorn\main.py", line 579, in run
    server.run()
  File "c:\Users\avina\AppData\Local\Programs\Python\Python312\Lib\site-packages\uvicorn\server.py", line 66, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\avina\AppData\Local\Programs\Python\Python312\Lib\site-packages\nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\avina\AppData\Local\Programs\Python\Python312\Lib\site-packages\nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "c:\Users\avina\Ap

In [115]:
app = FastAPI(title="HemoScan AI - Hybrid Multi-LLM Backend")

class PatientData(BaseModel):
    name: str
    phone: str
    Gender: str
    Age: float
    Hb: float
    RBC: float
    PCV: float
    MCV: float
    MCH: float
    MCHC: float


@app.get("/")
def home():
    return {"status": "HemoScan AI Backend Running with ML + Multi-LLM"}


@app.post("/predict")
def predict(data: PatientData):
    
    gender = 1 if data.Gender.lower() == "m" else 0
    
    
    input_data = np.array([[
        gender,
        data.Age,
        data.Hb,
        data.RBC,
        data.PCV,
        data.MCV,
        data.MCH,
        data.MCHC
        
        
    ]])
    
    
    prediction = model.predict(input_data)[0]
    prob = model.predict_proba(input_data)[0][1]
    confidence = float(prob * 100)
    
    result = "Anemic" if prediction == 1 else "Not Anemic"
    
    
    patient_dict = data.dict()
    
    
    
    prompt = build_medical_prompt(patient_dict, prediction, confidence)
    ai_report = gemini_report(prompt)
    pdf_path = generate_pdf_report(ai_report,"clinincal_report.pdf")

    return FileResponse(
        path=pdf_path,
        media_type="application/pdf",
        filename="HemoScan_AI_Report.pdf"
    )

In [116]:


uvicorn.run(app, host="0.0.0.0", port=8000)
  

INFO:     Started server process [23632]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:1861 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:1861 - "GET /openapi.json HTTP/1.1" 200 OK


c:\Users\avina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\avina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


INFO:     127.0.0.1:28023 - "POST /predict HTTP/1.1" 200 OK


c:\Users\avina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\avina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


INFO:     127.0.0.1:21101 - "POST /predict HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [23632]
